<a href="https://colab.research.google.com/github/mateocgonzalez07/TOI-256/blob/main/4_LHS1140_transitos_MCMC_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LHS 1140 b y c: radios a partir de la fotometría de TESS (PyTransit + MCMC)

**Objetivo:** ajustar los tránsitos de LHS 1140 b y c con fotometría pública de TESS y obtener sus radios, para combinarlos con las masas que salen de las velocidades radiales de HARPS (Problema 1, Ciencias Planetarias).

**Flujo:** Lightkurve/MAST, limpieza y aplanado (con los tránsitos enmascarados), conteo de tránsitos de b y c con datos, BLS, curvas plegadas, PyTransit, MCMC con **emcee**, radio, masa, semieje, prueba de excentricidad (efecto fotoexcéntrico) y figura del sistema con la zona habitable.

Basado en `02_transit_photometry.ipynb` (WASP-21 b): el radio ya no se estima con el punto mínimo de flujo, sino con un modelo físico, propagando la incertidumbre de $R_\star$.

**Alcance:** se ajustan **b** (P = 24.7 d) y **c** (P = 3.78 d), cada uno por separado con todos sus tránsitos. Los tránsitos del otro planeta se enmascaran, tanto en el aplanado como en la ventana de ajuste. No es un ajuste simultáneo ni un análisis de TTV.

Los parámetros publicados (Cadieux et al. 2024) son valores iniciales y ventanas de búsqueda, no resultados impuestos al ajuste. Requiere internet para descargar TESS desde MAST.

**Resultados principales:** Rp(b) = 1.78 R⊕ y Rp(c) = 1.25 R⊕; masas con los K del análisis de RV: Mp(b) = 7.0 ± 0.7 M⊕ (+2.0σ respecto a la literatura, 5.60 ± 0.19) y Mp(c) = 1.96 ± 0.37 M⊕ (+0.1σ). Los semiejes coinciden con la literatura. Con la excentricidad libre, TESS solo da límites al 95 % de 0.45 (b) y 0.44 (c), que no restringen nada frente a los de la literatura (< 0.043 y < 0.050).

## 0. Instalación e importaciones

Ejecuta una vez en el entorno del notebook.

In [ ]:
%pip install -qU numpy scipy pandas matplotlib astropy lightkurve pytransit emcee corner

### Importamos librerías

In [ ]:
import warnings
from pathlib import Path

import astropy.units as u
import corner
import emcee
import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Patch, Wedge
from pytransit import QuadraticModel

warnings.filterwarnings("ignore")
%matplotlib inline


## 1. Configuración del sistema

LHS 1140 (TIC 92226327, TOI-256) tiene dos planetas transitantes: b (P = 24.7 d) y c (P = 3.78 d). Los tiempos de tránsito de la Tabla D1 de Cadieux et al. (2024) vienen en BJD − 2 457 000, es decir, ya están en BTJD. Los parámetros sirven como semillas y priors amplios.

**a_rs** se cálculo dividiendo el semieje mayor (en ua) entre $R_\star$: el articulo no imprime a/R★. Con $R_\star$ = 0.2159 R☉, b da unos 94.2 y c unos 26.9.

**fit_names** es la lista de planetas que se ajustan con MCMC (b y c). **planet_names** es la lista para BLS y enmascarado.

### Constantes del sistema

Valores de LHS 1140 tomados de Cadieux et al. (2024). Es el mismo bloque en los notebooks de RV, tránsitos y zona habitable.

In [ ]:
# ============================================================================
# Constantes de LHS 1140 (Cadieux et al. 2024, arXiv:2310.15490)
# Mismo bloque en los notebooks de RV, tránsitos y zona habitable.
# Si cambias un valor aquí, cámbialo también en los otros dos.
# ============================================================================
P_ORB   = {"b": 24.73723, "c": 3.777940}        # d, periodo orbital (Tabla D1)
T0_BTJD = {"b": 1399.9300, "c": 1389.2939}      # BJD - 2457000, tiempo de tránsito (Tabla D1)
M_STAR, M_STAR_ERR = 0.1844, 0.0045             # M_sol (Tabla C1)
R_STAR, R_STAR_ERR = 0.2159, 0.0030             # R_sol (Tabla C1)
TEFF_STAR = 3096.0                              # K (Tabla C1)
P_ROT = 131.0                                   # d, periodo de rotación (Tabla C1)


In [ ]:
TARGET = "TIC 92226327"  # LHS 1140 (TOI-256) en TESS
BTJD_OFFSET = 2457000.0

# cadieux et al. (2024), Tabla C1
RSTAR_RSUN, RSTAR_ERR_RSUN = R_STAR, R_STAR_ERR      # del bloque de constantes
MSTAR_MSUN, MSTAR_ERR_MSUN = M_STAR, M_STAR_ERR

# cadieux et al. (2024): P y T0 de la Tabla D1 (T0 en BJD-2457000 = BTJD),
# b (parametro de impacto) de la Tabla D1, Rp/R* de la Tabla 1, radios del resumen del articulo. a_rs calculado (ver arriba).
# mask_dur = ancho (d) de la mascara de transito para el aplanado (~1.7 x T14), valor elegido en este trabajo.
catalog = {
    "b": {"period": P_ORB["b"], "t0_btjd": T0_BTJD["b"],
          "a_rs": 94.2, "b_impact": 0.23, "k": 0.0733,
          "rp_rearth_literature": 1.730, "mask_dur": 0.15},
    "c": {"period": P_ORB["c"], "t0_btjd": T0_BTJD["c"],
          "a_rs": 26.9, "b_impact": 0.09, "k": 0.0539,
          "rp_rearth_literature": 1.272, "mask_dur": 0.10}
}

planet_names = list(catalog.keys())  # para BLS y enmascarado
fit_names = ["b", "c"]  # planetas que se ajustan con MCMC

pd.DataFrame(catalog).T[[
    "period", "t0_btjd", "a_rs", "b_impact", "k",
    "rp_rearth_literature"
]]

## 2. Descargar curvas de luz de TESS

Se intenta primero recuperar productos SPOC de 120 segundos. Si no hay productos con esa cadencia, se buscan otras cadencias SPOC. No se selecciona un índice fijo del resultado: se inspeccionan y descargan los productos encontrados.

In [ ]:
search_result = lk.search_lightcurve(
    TARGET, mission="TESS", author="SPOC", exptime=120
)

if len(search_result) == 0:
    print("No hay productos SPOC de 120 s; buscando otras cadencias.")
    search_result = lk.search_lightcurve(
        TARGET, mission="TESS", author="SPOC"
    )

if len(search_result) == 0:
    raise RuntimeError("MAST no devolvio curvas SPOC para LHS 1140.")

display(search_result.table)

### Descargar los productos

In [ ]:
lc_collection = search_result.download_all()

if lc_collection is None or len(lc_collection) == 0:
    raise RuntimeError("No se pudieron descargar curvas de luz.")

print(f"Productos descargados: {len(lc_collection)}")
lc_collection

### Guardar copias FITS originales

In [ ]:
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw" / TARGET
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / TARGET
RESULTS_DIR = PROJECT_DIR / "data" / "results" / TARGET

for folder in (RAW_DIR, PROCESSED_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

for j, one_lc in enumerate(lc_collection):
    sector = one_lc.meta.get("SECTOR", j)
    out = RAW_DIR / f"{TARGET}_sector_{sector}.fits"
    one_lc.to_fits(str(out), overwrite=True)
    print("Guardado:", out)

## 3. Inspección y preprocesamiento

Se conserva el método del notebook original: **stitch**, **flatten** y eliminacion de outliers. Se utiliza un umbral conservador para reducir el riesgo de eliminar puntos reales del tránsito. El aplanado puede modificar la profundidad o duración si la ventana del filtro es demasiado corta; se debe comprobar la sensibilidad de los resultados a esta eleccion.

Aquí el aplanado se hace con los tránsitos de b y c enmascarados (**mask=**), para que el filtro no los trate como tendencia.

In [ ]:
lc_stitched = lc_collection.stitch().remove_nans()

# enmascarar los transitos conocidos (b y c) para que el aplanado no recorte su profundidad
mask_transitos = lc_stitched.create_transit_mask(
    period=[catalog[p]["period"] for p in planet_names],
    transit_time=[catalog[p]["t0_btjd"] for p in planet_names],
    duration=[catalog[p]["mask_dur"] for p in planet_names]
)

lc = (
    lc_stitched
    .flatten(window_length=901, break_tolerance=5, mask=mask_transitos)
    .remove_outliers(sigma_lower=8, sigma_upper=8)
    .remove_nans()
)

print("Numero de puntos:", len(lc))
print("Intervalo BTJD:", float(np.nanmin(lc.time.value)),
      "a", float(np.nanmax(lc.time.value)))

ax = lc.plot(color="midnightblue", linewidth=0.6)
ax.set_title(f"{TARGET} — TESS stitched and flattened light curve")
ax.set_xlabel("Time (BTJD)")
ax.set_ylabel("Normalized flux")
plt.show()

### Extraer arrays y revisar la cadencia

In [ ]:
time = np.asarray(lc.time.value, dtype=float)
flux = np.asarray(lc.flux.value, dtype=float)
flux_err = np.asarray(lc.flux_err.value, dtype=float)

valid = (np.isfinite(time) & np.isfinite(flux) &
         np.isfinite(flux_err) & (flux_err > 0))
time, flux, flux_err = time[valid], flux[valid], flux_err[valid]

order = np.argsort(time)
time, flux, flux_err = time[order], flux[order], flux_err[order]

exptime_days = float(lc.meta.get("TIMEDEL", 2 / 1440))  # cadencia en dias (2 minutos si no esta en el header)

# cadencia corta: 1 muestra. Cadencia larga: supersampling.
nsamples = 1 if exptime_days < 0.005 else 15

print(f"Exposure time: {exptime_days:.6f} d")
print(f"Supersampling: {nsamples}")
print(f"Median photometric uncertainty: {np.nanmedian(flux_err):.3e}")

## 4. Conteo de tránsitos de b y c: ¿cuántos tienen datos?

Con P = 24.7 d y sectores de ~27 d, b tendrá como mucho 1 o 2 tránsitos por sector, y algunos pueden caer en el hueco entre órbitas. c (P = 3.78 d) tendrá bastantes más. Hay que contarlos antes de ajustar.

Se cuentan los puntos a ±0.05 d de cada tránsito esperado (caben 72 de 120 s). Un tránsito se considera **completo** si tiene al menos 65 puntos (~90 %) y **parcial** si tiene menos.

In [ ]:
# conteo de transitos, elaborado con ayuda de Claude Sonnet 5.5 (Anthropic)
UMBRAL_COMPLETO = 65   # ~90 % de los 72 puntos de 120 s que caben en +-0.05 d
t_ini = np.nanmin(time)
t_fin = np.nanmax(time)

for planeta in fit_names:
    P = catalog[planeta]["period"]
    T0 = catalog[planeta]["t0_btjd"]

    n_min = int(np.floor((t_ini - T0) / P))
    n_max = int(np.ceil((t_fin - T0) / P))

    tn_con_datos = []
    puntos = []
    for n in range(n_min, n_max + 1):
        tn = T0 + n * P
        cuenta = np.sum(np.abs(time - tn) < 0.05)
        if cuenta > 0:
            tn_con_datos.append(tn)
            puntos.append(int(cuenta))

    print(planeta, ":", len(tn_con_datos), "transitos con datos")
    print("puntos por transito:", puntos)
    if len(puntos) > 0:
        print("promedio de puntos:", np.mean(puntos))
        n_comp = sum(p >= UMBRAL_COMPLETO for p in puntos)
        print(f"completos (>= {UMBRAL_COMPLETO} puntos): {n_comp} | parciales: {len(puntos) - n_comp}")

## 5. Búsqueda BLS de b y c

Se calcula un periodograma BLS por planeta dentro de una ventana de 0.5 % alrededor de su periodo de referencia. Esto refina señales conocidas; no es una búsqueda ciega de planetas desconocidos.

In [ ]:
bls_results = {}
periodograms = {}

# trabajamos con arrays NumPy sin unidades
# tiempo en BTJD (dias), flujo normalizado.

bls_model = BoxLeastSquares(
    time,
    flux,
    dy=flux_err
)

# duraciones de transito en dias
durations_days = np.linspace(0.035, 0.18, 30)

Para cada planeta se corre el BLS en una ventana de +-0.5 % alrededor de su periodo de referencia y se guarda el mejor periodo, el T0 y la profundidad

In [ ]:
for planet in planet_names:

    p_ref = catalog[planet]["period"]

    periods_days = np.linspace(
        p_ref * 0.995,
        p_ref * 1.005,
        2500
    )

    result = bls_model.power(
        periods_days,
        durations_days,
        objective="likelihood"
    )

    best_index = int(np.argmax(result.power))

    periodograms[planet] = {
        "period": np.asarray(result.period),
        "power": np.asarray(result.power)
    }

    bls_results[planet] = {
        "period": float(result.period[best_index]),
        "t0": float(result.transit_time[best_index]),
        "duration": float(result.duration[best_index]),
        "depth": float(result.depth[best_index])
    }

    r = bls_results[planet]

    print(
        f"LHS 1140 {planet}: "
        f"P={r["period"]:.8f} d, "
        f"duration={r["duration"]*24:.2f} h, "
        f"depth={r["depth"]*1e3:.3f} ppt"
    )

### Periodogramas BLS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, planet in zip(axes.flat, planet_names):

    pg = periodograms[planet]

    ax.plot(
        pg["period"],
        pg["power"],
        color="midnightblue",
        lw=1
    )

    ax.axvline(
        catalog[planet]["period"],
        color="crimson",
        ls="--",
        label="Periodo de referencia"
    )

    ax.set_title(f"LHS 1140 {planet}")
    ax.set_xlabel("Period (days)")
    ax.set_ylabel("BLS power")
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Curvas plegadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, planet in zip(axes.flat, planet_names):
    p = bls_results[planet]["period"]
    t0 = bls_results[planet]["t0"]
    folded = lc.fold(period=p * u.day, epoch_time=t0 * u.day)

    ax.scatter(folded.phase.to_value(u.day), folded.flux.value,
               s=3, alpha=0.35, color="midnightblue")
    ax.set_xlim(-0.25, 0.25)
    ax.set_title(f"LHS 1140 {planet}: P={p:.6f} d")
    ax.set_xlabel("Time from transit (days)")
    ax.set_ylabel("Normalized flux")

plt.tight_layout()
plt.show()

## 7. Modelo físico con PyTransit

Se utiliza **QuadraticModel**, que implementa el modelo analítico de tránsito con oscurecimiento al limbo cuadrático:

$$
I(\mu)=1-u_1(1-\mu)-u_2(1-\mu)^2.
$$

Para una órbita circular:

$$
  b = \frac{a}{R_{star}} \cos i
$$

$$
i = \arccos\left( \frac{b}{a/R_{star}} \right).
$$

Los coeficientes de limb darkening vienen de la solución de TESS de Cadieux et al. (2024, Tabla D2): q1 = 0.33 y q2 = 0.56, convertidos a u1 = 0.643 y u2 = −0.069 (Kipping 2013). Se fijan; una mejora sería ajustarlos con priors.

In [ ]:
# cadieux et al. (2024), Tabla D2, TESS: q1 = 0.33, q2 = 0.56 (Kipping 2013)
# u1 = 2*sqrt(q1)*q2 = 0.643 ; u2 = sqrt(q1)*(1 - 2*q2) = -0.069
LD_COEFF = np.array([0.643, -0.069])

def make_transit_model(t):
    model = QuadraticModel()
    model.set_data(
        np.asarray(t, dtype=float),
        exptimes=exptime_days,
        nsamples=nsamples
    )
    return model


def evaluate_planet_transit(model, theta):
    # theta = [k, t0, period, a/Rstar, impact, log_jitter]
    k, t0, period, a_rs, impact, log_jitter = theta
    cos_i = np.clip(impact / a_rs, -1.0, 1.0)
    inclination = np.arccos(cos_i)

    return model.evaluate(
        k=k, ldc=LD_COEFF, t0=t0, p=period,
        a=a_rs, i=inclination
    )

## 8. Seleccionar ventanas de tránsito

Para ajustar cada planeta se usan las observaciones próximas a todos sus tránsitos. Se excluyen además las ventanas donde se esperan tránsitos de los otros planetas, para que sus señales no contaminen el ajuste individual.

In [ ]:
def wrapped_time_distance(t, period, t0):
    return (t - t0 + 0.5 * period) % period - 0.5 * period


def epoch_near_data(t0, period, time_array):
    t_mid = np.nanmedian(time_array)
    n = np.round((t_mid - t0) / period)
    return t0 + n * period


def select_planet_data(planet, half_window=0.18, other_planet_mask=0.12):
    p0 = bls_results[planet]["period"]
    t0_bls = bls_results[planet]["t0"]
    t0_ref = epoch_near_data(t0_bls, p0, time)

    own_distance = np.abs(wrapped_time_distance(time, p0, t0_ref))
    mask = own_distance < half_window

    for other in planet_names:
        if other == planet:
            continue
        p_other = bls_results[other]["period"]
        t0_other = epoch_near_data(bls_results[other]["t0"], p_other, time)
        other_distance = np.abs(
            wrapped_time_distance(time, p_other, t0_other)
        )
        mask &= other_distance > other_planet_mask

    return time[mask], flux[mask], flux_err[mask], t0_ref


for planet in planet_names:
    tp, fp, ep, t0ref = select_planet_data(planet)
    print(f"Planet {planet}: {len(tp)} points; reference epoch BTJD={t0ref:.6f}")

## 9. Inferencia bayesiana

Para cada planeta se estiman seis parámetros:

- $k=R_p/R_\star$: radio relativo.
- $T_0$: epoca de tránsito de referencia (BTJD).
- $P$: periodo orbital (dias).
- $a/R_\star$: semieje mayor relativo.
- $b$: parámetro de impacto.
- $\ln\sigma_{\rm jit}$: logaritmo del ruido adicional.

Se usa:

$$
p(\theta\mid D)\propto\mathcal{L}(D\mid\theta)p(\theta),
\qquad
\sigma_i^2=\sigma_{\mathrm{phot},i}^2+\sigma_{\mathrm{jit}}^2.
$$

Los priors son uniformes dentro de límites físicos y ventanas alrededor de los valores de referencia.

In [ ]:
def log_prior(theta, planet, t0_ref, p_ref):
    k, t0, period, a_rs, impact, log_jitter = theta
    a0 = catalog[planet]["a_rs"]

    if not (0.003 < k < 0.12):
        return -np.inf
    if not (abs(t0 - t0_ref) < 0.10):
        return -np.inf
    if not (abs(period - p_ref) < 0.005 * p_ref):
        return -np.inf
    if not (0.5 * a0 < a_rs < 1.5 * a0):
        return -np.inf
    if not (0.0 <= impact < 1.0 + k):
        return -np.inf

    jitter = np.exp(log_jitter)
    if not (1e-7 < jitter < 0.02):
        return -np.inf

    return 0.0

### Log-probabilidad
Prior uniforme y verosimilitud gaussiana con ruido extra (jitter). Si el modelo no es finito, el punto se descarta

In [ ]:
def log_probability(theta, planet, t, f, ferr, model, t0_ref, p_ref):
    lp = log_prior(theta, planet, t0_ref, p_ref)
    if not np.isfinite(lp):
        return -np.inf

    model_flux = evaluate_planet_transit(model, theta)
    if not np.all(np.isfinite(model_flux)):
        return -np.inf

    jitter = np.exp(theta[-1])
    variance = ferr**2 + jitter**2
    residuals = f - model_flux

    loglike = -0.5 * np.sum(
        residuals**2 / variance + np.log(2.0 * np.pi * variance)
    )
    return loglike + lp

## 10. Ejecutar MCMC

Se utiliza **emcee** con 32 caminantes inicializados cerca de los valores BLS y de referencia. La primera corrida es de 12000 pasos; como no alcanzó 50 veces el tiempo de autocorrelación, se continúa en la sección 11.

In [ ]:
NWALKERS = 32
NSTEPS = 12000
BURNIN = 2000
THIN = 50

# nombres de los parametros del ajuste (se usan en los diagnosticos y en el resumen)
PARAMETER_NAMES = [
    "k = Rp/Rstar", "T0 (BTJD)", "Period (days)",
    "a/Rstar", "Impact parameter", "ln(jitter)"
]


### Función del ajuste
Función que ajusta un planeta con emcee: inicializa los caminantes cerca de los valores del BLS y del catalogo, corre la cadena y devuelve las muestras

In [ ]:
def fit_one_planet(planet, random_seed=42):
    t_fit, f_fit, err_fit, t0_ref = select_planet_data(planet)

    if len(t_fit) < 30:
        raise RuntimeError(f"Muy pocos puntos para LHS 1140 {planet}: {len(t_fit)}")

    p_ref = bls_results[planet]["period"]
    k0 = catalog[planet]["k"]
    a0 = catalog[planet]["a_rs"]
    b0 = catalog[planet]["b_impact"]

    tm = make_transit_model(t_fit)

    center = np.array([k0, t0_ref, p_ref, a0, b0, np.log(3e-4)])
    scales = np.array([
        max(k0 * 0.04, 2e-4),
        0.002,
        p_ref * 2e-5,
        max(a0 * 0.01, 0.05),
        0.015,
        0.25
    ])

    rng = np.random.default_rng(random_seed)
    pos = center + scales * rng.normal(size=(NWALKERS, len(center)))

    for j in range(NWALKERS):
        attempts = 0
        while not np.isfinite(log_prior(pos[j], planet, t0_ref, p_ref)):
            pos[j] = center + scales * rng.normal(size=len(center))
            attempts += 1
            if attempts > 10000:
                raise RuntimeError("No fue posible inicializar los caminantes.")

    sampler = emcee.EnsembleSampler(
        NWALKERS, len(center), log_probability,
        args=(planet, t_fit, f_fit, err_fit, tm, t0_ref, p_ref)
    )
    sampler.run_mcmc(pos, NSTEPS, progress=True)

    samples = sampler.get_chain(
        discard=BURNIN, thin=THIN, flat=True
    )

    return {
        "planet": planet, "time": t_fit, "flux": f_fit,
        "flux_err": err_fit, "model": tm, "sampler": sampler,
        "samples": samples, "t0_ref": t0_ref, "p_ref": p_ref
    }

### Ejecutar el ajuste para b y c
Esta celda puede tardar. Para una prueba inicial reduce **NSTEPS**; para resultados finales aumenta la longitud de las cadenas y verifica convergencia.

In [ ]:
fits = {}

for j, planet in enumerate(fit_names):
    print(f"LHS 1140 {planet}")
    fits[planet] = fit_one_planet(planet, random_seed=42 + j)

print("Ajustes terminados:", list(fits.keys()))

## 11. Diagnósticos MCMC y continuación de las cadenas

La fracción de aceptación y las trazas son diagnósticos, no pruebas aisladas de convergencia. Después se alargan las cadenas y se calcula el tiempo de autocorrelación. Todo lo que sigue (resumen, corner plots, radios y masas) usa las cadenas ya alargadas.

In [ ]:
for planet in fit_names:
    sampler = fits[planet]["sampler"]
    print(
        f"LHS 1140 {planet}: mean acceptance fraction = "
        f"{np.mean(sampler.acceptance_fraction):.3f}"
    )

### Trazas de las cadenas

In [ ]:
for planet in fit_names:
    chain = fits[planet]["sampler"].get_chain()
    fig, axes = plt.subplots(
        chain.shape[2], 1, figsize=(11, 10), sharex=True
    )
    for i, ax in enumerate(axes):
        ax.plot(chain[:, :, i], alpha=0.25, lw=0.5)
        ax.set_ylabel(PARAMETER_NAMES[i])
    axes[-1].set_xlabel("MCMC step")
    fig.suptitle(f"LHS 1140 {planet} — MCMC traces", y=1.01)
    plt.tight_layout()
    plt.show()

### Continuar las cadenas
La primera corrida no cumplía 50 veces el tiempo de autocorrelación. Se alargan las cadenas 13000 pasos desde donde quedaron (25000 en total) y se recalculan las muestras, descartando 8000 pasos de burn-in (unas 17 veces el τ máximo, ~470 pasos). Las celdas de resultados que vienen después usan estas muestras.

In [ ]:
for planet in fit_names:
    fits[planet]["sampler"].run_mcmc(None, 13000, progress=True)
    fits[planet]["samples"] = fits[planet]["sampler"].get_chain(
        discard=8000, thin=100, flat=True
    )

### Tiempos de autocorrelación

In [ ]:
for planet in fit_names:
    sampler = fits[planet]["sampler"]
    print(f"\nLHS 1140 {planet}")
    tau = sampler.get_autocorr_time(tol=0)
    for name, value in zip(PARAMETER_NAMES, tau):
        print(f"{name}: tau = {value:.1f}")
    print("Steps / max(tau):", sampler.iteration / np.max(tau))

## 12. Resumen de las distribuciones posteriores
Se reportan la mediana y los percentiles 16 y 84; no se presupone que las posteriores sean gaussianas.

In [ ]:
summary_rows = []
for planet in fit_names:
    samples = fits[planet]["samples"]
    row = {"planet": planet}
    for i, name in enumerate(PARAMETER_NAMES):
        p16, p50, p84 = np.percentile(samples[:, i], [16, 50, 84])
        row[name] = p50
        row[name + " lower"] = p50 - p16
        row[name + " upper"] = p84 - p50
    summary_rows.append(row)

posterior_summary = pd.DataFrame(summary_rows).set_index("planet")
display(posterior_summary)

## 13. Corner plots

In [ ]:
for planet in fit_names:
    samples = fits[planet]["samples"]
    fig = corner.corner(
        samples,
        labels=[
            r"$R_p/R_\star$", r"$T_0$ (BTJD)", r"$P$ (d)",
            r"$a/R_\star$", r"$b$", r"$\ln\sigma_{\rm jit}$"
        ],
        quantiles=[0.16, 0.50, 0.84],
        show_titles=True,
        title_fmt=".6f"
    )
    fig.suptitle(f"LHS 1140 {planet} — posterior distributions", y=1.02)
    plt.show()

## 14. Modelo final y residuos

In [ ]:
final_models = {}

for planet in fit_names:
    result = fits[planet]
    theta_med = np.median(result["samples"], axis=0)
    model_flux = evaluate_planet_transit(result["model"], theta_med)

    final_models[planet] = {
        "theta_median": theta_med,
        "model_flux": model_flux,
        "residuals": result["flux"] - model_flux
    }

    t_rel = result["time"] - theta_med[1]
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(10, 6), sharex=True,
        gridspec_kw={"height_ratios": [3, 1]}
    )

    ax1.errorbar(t_rel, result["flux"], yerr=result["flux_err"],
                 fmt=".", ms=3, alpha=0.45, label="TESS")
    ax1.plot(t_rel, model_flux, color="crimson", lw=2,
             label="PyTransit + MCMC")
    ax1.set_ylabel("Normalized flux")
    ax1.set_title(f"LHS 1140 {planet}: transit fit")
    ax1.legend()

    ax2.axhline(0, color="black", ls="--", lw=1)
    ax2.errorbar(t_rel, result["flux"] - model_flux,
                 yerr=result["flux_err"], fmt=".", ms=3, alpha=0.45)
    ax2.set_xlabel("Time from fitted transit epoch (days)")
    ax2.set_ylabel("Residuals")

    plt.tight_layout()
    plt.show()

## 15. Convertir $(R_p/R_{star})$ en radio planetario

$$
R_p = (R_p/R_{star})R_{star}.
$$

Se propagan tanto las muestras posteriores de $k$ como la incertidumbre del radio estelar mediante realizaciones Monte Carlo. Se usa $R_\odot/R_\oplus\simeq109.076$.

In [ ]:
RSUN_TO_REARTH = 109.076  # radio solar / radio terrestre, iau 2015
physical_rows = []

for j, planet in enumerate(fit_names):
    samples = fits[planet]["samples"]
    k_samples = samples[:, 0]

    rng = np.random.default_rng(100 + j)
    rstar_draws = rng.normal(
        RSTAR_RSUN, RSTAR_ERR_RSUN, size=len(k_samples)
    )
    rstar_draws = np.clip(rstar_draws, 0.01, None)

    rp_draws = k_samples * rstar_draws * RSUN_TO_REARTH
    p16, p50, p84 = np.percentile(rp_draws, [16, 50, 84])

    physical_rows.append({
        "planet": planet,
        "Rp/Rstar median": np.median(k_samples),
        "Rp median (R_earth)": p50,
        "Rp lower (R_earth)": p50 - p16,
        "Rp upper (R_earth)": p84 - p50,
        "Rp literature (R_earth)": catalog[planet]["rp_rearth_literature"]
    })

physical_results = pd.DataFrame(physical_rows).set_index("planet")
display(physical_results)

## 16. Comparar con la literatura
La comparación es un control externo, no un ajuste de los resultados hacia los valores publicados.

In [ ]:
comparison = physical_results[[
    "Rp median (R_earth)", "Rp literature (R_earth)"
]].copy()

comparison["relative difference (%)"] = (
    100 * (
        comparison["Rp median (R_earth)"] -
        comparison["Rp literature (R_earth)"]
    ) / comparison["Rp literature (R_earth)"]
)

display(comparison)

ax = comparison[[
    "Rp median (R_earth)", "Rp literature (R_earth)"
]].plot(kind="bar", figsize=(10, 5))
ax.set_ylabel(r"Planet radius ($R_\oplus$)")
ax.set_xlabel("Planet")
ax.set_title("LHS 1140: posterior radii vs literature")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 17. Guardar resultados

In [ ]:
summary_file = RESULTS_DIR / "posterior_summary.csv"
physical_file = RESULTS_DIR / "planetary_radii.csv"

posterior_summary.to_csv(summary_file)
physical_results.to_csv(physical_file)

for planet in fit_names:
    np.save(
        RESULTS_DIR / f"LHS1140_{planet}_posterior_samples.npy",
        fits[planet]["samples"]
    )

print("Resultados guardados en:", RESULTS_DIR)
print(summary_file)
print(physical_file)

## 18. Masas de b y c a partir de las velocidades radiales

$$
M_p \sin i = K\,\left(\frac{P}{2\pi G}\right)^{1/3} M_\star^{2/3}
$$

(órbita circular y $M_p \ll M_\star$). Datos (análisis de RV): $K_b \approx 4.8 \pm 0.45$ m/s y $K_c = 2.47 \pm 0.47$ m/s; $P$ y $M_\star$ de la configuración. Con la inclinación del ajuste, $\cos i = b / (a/R_\star)$, se obtiene $M_p = M_p\sin i / \sin i$.

**Comprobación:** con el K del análisis de RV, b debe dar unos 7.0 ± 0.7 M⊕ y c unos 1.96 ± 0.37 M⊕. Con los K de la literatura (3.80 y 2.42 m/s) salen unos 5.6 y 1.9 M⊕.

In [ ]:
# G: codata 2018. MSUN y MEARTH: iau 2015 (gm nominal / G). AU: iau 2012
G = 6.67430e-11
MSUN = 1.98841e30
MEARTH = 5.9722e24
DAY = 86400.0

# K del analisis de RV (notebook de velocidades radiales, ajuste de b y c): (valor, error) en m/s
# nota: el notebook de RV da K_b = 4.78; aqui se uso 4.77 (redondeo). Si cambias K, vuelve a correr todo desde esta celda.
K_RV = {"b": (4.77, 0.45), "c": (2.47, 0.47)}
# cadieux et al. (2024), resumen del articulo (masa en M_earth)
MP_LIT = {"b": (5.60, 0.19), "c": (1.91, 0.06)}

mass_results = {}
for j, planet in enumerate(fit_names):
    samples = fits[planet]["samples"]
    n = len(samples)
    rng = np.random.default_rng(200 + j)

    k_draw = rng.normal(K_RV[planet][0], K_RV[planet][1], size=n)
    ms_draw = rng.normal(MSTAR_MSUN, MSTAR_ERR_MSUN, size=n) * MSUN

    period = samples[:, 2] * DAY
    a_rs = samples[:, 3]
    impact = samples[:, 4]
    sin_i = np.sqrt(1.0 - np.clip(impact / a_rs, 0.0, 1.0)**2)

    msini = k_draw * (period / (2 * np.pi * G))**(1/3) * ms_draw**(2/3) / MEARTH
    mass = msini / sin_i

    p16, p50, p84 = np.percentile(mass, [16, 50, 84])
    mass_results[planet] = {"p16": p16, "p50": p50, "p84": p84}

    lit, e_lit = MP_LIT[planet]
    err = 0.5 * (p84 - p16)
    dif = (p50 - lit) / np.hypot(err, e_lit)
    print(f"LHS 1140 {planet}: Mp = {p50:.2f} +{p84 - p50:.2f} -{p50 - p16:.2f} M_earth"
          f"  | literatura {lit:.2f} +- {e_lit:.2f}  | diferencia {dif:+.1f} sigma"
          f"  | sin i medio = {np.mean(sin_i):.5f}")

## 19. Semieje mayor
Se calcula el semieje mayor de cada planeta con la tercera ley de Kepler y se compara con la literatura (Cadieux et al. 2024, Tabla D1)

In [ ]:
AU = 1.495978707e11
A_LIT = {"b": (0.0946, 0.0017), "c": (0.0270, 0.0005)}  # cadieux et al. (2024), Tabla D1 (ua)

for j, planet in enumerate(fit_names):
    samples = fits[planet]["samples"]
    rng = np.random.default_rng(300 + j)
    ms = rng.normal(MSTAR_MSUN, MSTAR_ERR_MSUN, size=len(samples)) * MSUN
    period = samples[:, 2] * DAY
    a_au = (G * ms * period**2 / (4 * np.pi**2))**(1/3) / AU

    p16, p50, p84 = np.percentile(a_au, [16, 50, 84])
    lit, e_lit = A_LIT[planet]
    dif = (p50 - lit) / np.hypot(0.5 * (p84 - p16), e_lit)
    print(f"LHS 1140 {planet}: a = {p50:.4f} +{p84 - p50:.4f} -{p50 - p16:.4f} ua"
          f"  | literatura {lit:.4f} +- {e_lit:.4f}  | diferencia {dif:+.1f} sigma")

## 20. Efecto fotoexcéntrico

Ajuste con excentricidad y omega libres y con prior de densidad estelar, para ver si cambia el resultado del circular. Usa 40 caminantes y 12000 pasos (burn-in 3000), así que su convergencia es menor que la del ajuste circular (ver resultados).

In [ ]:
# efecto fotoexcentrico: ajuste con e y omega libres + prior de densidad estelar
# parametros: [k, T0, P, b, sqrt(e)cos(w), sqrt(e)sin(w), rho_star, ln(jitter)]
# a/R* no es libre: sale de rho_star y P (tercera ley de Kepler). El resto del flujo es el del ajuste circular.
RHO_SUN = 1.410  # g/cm^3, densidad media del sol
RHO_STAR = RHO_SUN * MSTAR_MSUN / RSTAR_RSUN**3  # densidad de la estrella (g/cm^3)
RHO_ERR = RHO_STAR * np.hypot(MSTAR_ERR_MSUN / MSTAR_MSUN, 3 * RSTAR_ERR_RSUN / RSTAR_RSUN)
E_MAX = 0.5  # prior: e uniforme en [0, E_MAX]
G_CGS = 6.6743e-8  # G en cgs (codata 2018)
print(f"rho_star = {RHO_STAR:.1f} +- {RHO_ERR:.1f} g/cm^3")

NW_E, NSTEPS_E, BURNIN_E, THIN_E = 40, 12000, 3000, 10
ECC_NAMES = ["k", "T0", "P", "b", "sqrt(e)cos(w)", "sqrt(e)sin(w)", "rho_star", "ln(jitter)"]

### Semieje desde la densidad estelar
En el ajuste excentrico a/R* no es libre: se calcula con la densidad de la estrella y el periodo (tercera ley de Kepler)

In [ ]:
def a_rs_desde_rho(rho, period_days):
    # a/R* = (G rho P^2 / (3 pi))^(1/3)   (cgs)
    return (G_CGS * rho * (period_days * 86400.0)**2 / (3.0 * np.pi))**(1.0 / 3.0)

### Modelo de tránsito con e y omega
Se evalua el modelo de PyTransit con la excentricidad y el argumento del periastro. La inclinación sale de b, e, omega y a/R*

In [ ]:
def evaluar_ecc(model, theta):
    k, t0, period, b, sq_c, sq_s, rho, log_jit = theta
    e = sq_c**2 + sq_s**2
    w = np.arctan2(sq_s, sq_c)
    a_rs = a_rs_desde_rho(rho, period)
    cos_i = b * (1.0 + e * np.sin(w)) / (a_rs * (1.0 - e**2))
    inc = np.arccos(np.clip(cos_i, -1.0, 1.0))
    return model.evaluate(k=k, ldc=LD_COEFF, t0=t0, p=period, a=a_rs, i=inc, e=e, w=w)

### Log-probabilidad con e libre
Priors uniformes, límite de e, y prior gaussiano para la densidad estelar, mas la verosimilitud gaussiana con jitter

In [ ]:
def log_prob_ecc(theta, t0_ref, p_ref, model, f, ferr):
    k, t0, period, b, sq_c, sq_s, rho, log_jit = theta
    e = sq_c**2 + sq_s**2
    if not (0.003 < k < 0.12): return -np.inf
    if not (abs(t0 - t0_ref) < 0.10): return -np.inf
    if not (abs(period - p_ref) < 0.005 * p_ref): return -np.inf
    if not (0.0 <= b < 1.0 + k): return -np.inf
    if not (e < E_MAX): return -np.inf
    if not (5.0 < rho < 80.0): return -np.inf
    if not (1e-7 < np.exp(log_jit) < 0.02): return -np.inf
    w = np.arctan2(sq_s, sq_c)
    a_rs = a_rs_desde_rho(rho, period)
    if b * (1.0 + e * np.sin(w)) / (a_rs * (1.0 - e**2)) >= 1.0: return -np.inf
    modelo = evaluar_ecc(model, theta)
    if not np.all(np.isfinite(modelo)):
        return -np.inf
    var = ferr**2 + np.exp(log_jit)**2
    loglike = -0.5 * np.sum((f - modelo)**2 / var + np.log(2.0 * np.pi * var))
    lp_rho = -0.5 * ((rho - RHO_STAR) / RHO_ERR)**2  # prior gaussiano de densidad estelar
    return loglike + lp_rho

### Función del ajuste con e libre
Mismo procedimiento que el ajuste circular, con los parámetros de e y omega y la densidad estelar

In [ ]:
def ajustar_ecc(planet, seed=7):
    t_fit, f_fit, err_fit, t0_ref = select_planet_data(planet)
    p_ref = bls_results[planet]["period"]
    tm = make_transit_model(t_fit)
    centro = np.array([catalog[planet]["k"], t0_ref, p_ref, catalog[planet]["b_impact"],
                       0.05, 0.05, RHO_STAR, np.log(3e-4)])
    escala = np.array([max(catalog[planet]["k"] * 0.03, 2e-4), 0.001, p_ref * 1e-5, 0.05,
                       0.08, 0.08, 0.5 * RHO_ERR, 0.25])
    rng = np.random.default_rng(seed)
    pos = centro + escala * rng.normal(size=(NW_E, len(centro)))
    for j in range(NW_E):
        intentos = 0
        while not np.isfinite(log_prob_ecc(pos[j], t0_ref, p_ref, tm, f_fit, err_fit)):
            pos[j] = centro + escala * rng.normal(size=len(centro))
            intentos += 1
            if intentos > 10000:
                raise RuntimeError("No se pudo inicializar a los caminantes")
    sampler = emcee.EnsembleSampler(NW_E, len(centro), log_prob_ecc, args=(t0_ref, p_ref, tm, f_fit, err_fit))
    sampler.run_mcmc(pos, NSTEPS_E, progress=True)
    samples = sampler.get_chain(discard=BURNIN_E, thin=THIN_E, flat=True)
    return {"sampler": sampler, "samples": samples}

### Ejecutar el ajuste con e libre
Se corre para b y c. Tarda unos minutos por planeta

In [ ]:
fits_ecc = {}
for planet in fit_names:
    print(f"LHS 1140 {planet}: ajuste con e y omega libres")
    fits_ecc[planet] = ajustar_ecc(planet)

### Resultados del ajuste con e libre
Mediana de e y su límite superior al 95 %, densidad estelar posterior, radio con e libre y convergencia, comparados con la literatura (Cadieux et al. 2024)

In [ ]:
LIT_E95 = {"b": 0.043, "c": 0.050}  # cadieux et al. (2024), Tabla D1, limites al 95%
fig, ax = plt.subplots(1, len(fit_names), figsize=(5.5 * len(fit_names), 3.8), squeeze=False)
for a, planet in zip(ax[0], fit_names):
    sampler = fits_ecc[planet]["sampler"]
    s = fits_ecc[planet]["samples"]
    e = s[:, 4]**2 + s[:, 5]**2
    p16, p50, p84, p95 = np.percentile(e, [16, 50, 84, 95])
    tau = sampler.get_autocorr_time(tol=0)
    print(f"\nLHS 1140 {planet}:")
    print(f"  e = {p50:.3f} +{p84 - p50:.3f} -{p50 - p16:.3f} | limite superior 95% = {p95:.3f} (literatura: < {LIT_E95[planet]})")
    print(f"  rho_star posterior = {np.median(s[:, 6]):.1f} +- {np.std(s[:, 6]):.1f} g/cm^3 (prior {RHO_STAR:.1f} +- {RHO_ERR:.1f})")
    rp = np.median(s[:, 0]) * RSTAR_RSUN * RSUN_TO_REARTH
    print(f"  Rp = {rp:.3f} R_earth (con e libre)")
    print(f"  aceptacion = {np.mean(sampler.acceptance_fraction):.2f} | pasos/tau_max = {sampler.iteration / np.max(tau):.0f} (>50 es lo ideal)")
    a.hist(e, bins=40, color="C0", alpha=0.8)
    a.axvline(p95, color="C3", ls="-", label=f'95% = {p95:.2f}')
    a.axvline(LIT_E95[planet], color="k", ls="--", label=f'literatura: {LIT_E95[planet]}')
    a.set_xlabel("excentricidad e")
    a.set_ylabel("muestras")
    a.set_title(f'LHS 1140 {planet}')
    a.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 21. Sistema visto desde arriba y zona habitable

Órbitas circulares de b y c y zona habitable, con los radios y masas de las secciones anteriores. Requiere haber corrido antes las secciones de radios, masas y semieje.

In [ ]:
# sistema LHS 1140 visto desde arriba: orbitas circulares + zona habitable
# requiere haber corrido antes las celdas de radios, masas y semieje (physical_results, mass_results, G, MSUN, DAY, AU).

# TEFF_STAR (K) viene del bloque de constantes (Cadieux et al. 2024, Tabla C1)
L_STAR = RSTAR_RSUN**2 * (TEFF_STAR / 5772.0)**4  # L/L_sun a partir de R y Teff (Teff del sol = 5772 K, iau 2015)

### Coeficientes de la zona habitable
Coeficientes de Kopparapu et al. (2013), Tabla 3, para los bordes de la zona habitable de una estrella. La estrella usa Teff = 3096 K (Cadieux et al. 2024, Tabla C1).

In [ ]:
KOP = {  # kopparapu et al. (2013), Tabla 3 (valida para 2600 K <= Teff <= 7200 K): Seff_sun, a, b, c, d
    "Recent Venus": (1.7753, 1.4316e-4, 2.9875e-9, -7.5702e-12, -1.1635e-15),
    "Moist Greenhouse": (1.0140, 8.1774e-5, 1.7063e-9, -4.3241e-12, -6.6462e-16),
    "Maximum Greenhouse": (0.3438, 5.8942e-5, 1.6558e-9, -3.0045e-12, -5.2983e-16),
    "Early Mars": (0.3179, 5.4513e-5, 1.5313e-9, -2.7786e-12, -4.8997e-16),
}

### Bordes de la zona habitable
Función que calcula el borde (en ua) a partir de la temperatura efectiva y la luminosidad

In [ ]:
def borde_hz(nombre, teff, lum):
    s0, a, b, c, d = KOP[nombre]
    T = teff - 5780.0
    seff = s0 + a * T + b * T**2 + c * T**3 + d * T**4
    return np.sqrt(lum / seff)  # ua

### Zona habitable de LHS 1140
Bordes conservador y optimista de la zona habitable

In [ ]:
hz_in_opt = borde_hz("Recent Venus", TEFF_STAR, L_STAR)
hz_in_con = borde_hz("Moist Greenhouse", TEFF_STAR, L_STAR)
hz_out_con = borde_hz("Maximum Greenhouse", TEFF_STAR, L_STAR)
hz_out_opt = borde_hz("Early Mars", TEFF_STAR, L_STAR)
print(f"L = {L_STAR:.5f} L_sun | ZH conservadora: {hz_in_con:.4f} a {hz_out_con:.4f} ua | optimista: {hz_in_opt:.4f} a {hz_out_opt:.4f} ua")

### Datos de los planetas
Semieje, radio, masa e insolación de cada planeta, tomados de las celdas anteriores

In [ ]:
# datos de cada planeta (ya calculados en el notebook)
datos = {}
for planet in fit_names:
    P = np.median(fits[planet]["samples"][:, 2]) * DAY
    a_au = (G * MSTAR_MSUN * MSUN * P**2 / (4 * np.pi**2))**(1/3) / AU
    datos[planet] = dict(a=a_au,
                         rp=physical_results.loc[planet, "Rp median (R_earth)"],
                         mp=mass_results[planet]["p50"],
                         S=L_STAR / a_au**2)

### Gráfica del sistema
Sistema visto desde arriba: órbitas circulares y zona habitable. La posición de cada planeta en su órbita es solo para dibujar

In [ ]:
ANGULO = {"b": 50.0, "c": 200.0}  # grados: posicion en la orbita, solo para dibujar
LIM = hz_out_opt * 1.25

fig, ax = plt.subplots(figsize=(6.5, 7.5))
ax.add_patch(Wedge((0, 0), hz_out_opt, 0, 360, width=hz_out_opt - hz_in_opt, color="tab:green", alpha=0.15, lw=0))
ax.add_patch(Wedge((0, 0), hz_out_con, 0, 360, width=hz_out_con - hz_in_con, color="tab:green", alpha=0.30, lw=0))
ax.scatter([0], [0], s=350, color="orangered", zorder=4)

handles = [Patch(color="tab:green", alpha=0.15, label="zona habitable optimista"),
           Patch(color="tab:green", alpha=0.30, label="zona habitable conservadora"),
           Line2D([0], [0], marker="o", color="w", markerfacecolor="orangered", markersize=9,
                  label="LHS 1140 (tamaño no a escala)")]
for color, (planet, d) in zip(["C0", "C1"], datos.items()):
    ax.add_patch(Circle((0, 0), d["a"], fill=False, ls="--", lw=1, color="0.4", zorder=2))  # orbita circular
    ang = np.radians(ANGULO[planet])
    x, y = d["a"] * np.cos(ang), d["a"] * np.sin(ang)
    ax.scatter([x], [y], s=110 * d["rp"]**2, color=color, edgecolor="k", zorder=5)
    ax.annotate(planet, (x, y), xytext=(8, 8), textcoords="offset points", fontsize=12, weight="bold")
    handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markeredgecolor="k", markersize=9,
                          label=f"{planet}: a = {d["a"]:.4f} ua, Rp = {d["rp"]:.2f} R⊕, Mp = {d["mp"]:.1f} M⊕, S = {d["S"]:.2f} S⊕"))

ax.set_xlim(-LIM, LIM)
ax.set_ylim(-LIM, LIM)
ax.set_aspect("equal")
ax.set_xlabel("ua")
ax.set_ylabel("ua")
ax.set_title("Sistema LHS 1140")
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sistema_LHS1140_orbitas.png", dpi=150, bbox_inches="tight")
plt.show()

## 22. Conclusiones y limitaciones

Este notebook caracteriza los tránsitos de LHS 1140 b y c con fotometría de TESS. BLS refina el periodo; PyTransit modela la forma del tránsito; MCMC estima las distribuciones posteriores de $R_p/R_\star$, $T_0$, $P$, $a/R_\star$, $b$ y el ruido adicional. Cada radio se combina con la masa de las velocidades radiales para obtener la densidad.

### Limitaciones

1. **Pocos tránsitos en b:** con P = 24.7 d, el ajuste de b descansa en 3 tránsitos completos; c tiene 17 (13 completos y 4 parciales, con el criterio de ≥ 65 puntos), ver el conteo de la sección 4.
2. **Limb darkening:** fijado con los coeficientes de TESS de Cadieux et al. (2024); lo ideal sería ajustarlos con priors.
3. **Aplanado:** **flatten(window_length=901)** con los tránsitos enmascarados; repetir con otras ventanas y comprobar estabilidad.
4. **Ajuste independiente:** cada planeta se ajusta por separado y se enmascaran los tránsitos del otro. El siguiente nivel es un ajuste simultáneo.
5. **Órbita circular:** la relación entre $b$ e inclinación supone excentricidad cero. El ajuste con e libre (sección 20) da límites al 95 % de 0.45 (b) y 0.44 (c): TESS solo no restringe la excentricidad.
6. **Ruido correlacionado:** **jitter** es ruido blanco adicional; no sustituye un proceso gaussiano.
7. **Convergencia:** ajuste circular con 32 caminantes y 25000 pasos (12000 + 13000), burn-in de 8000. Steps / max(tau) = 53.5 para b y 53.4 para c: cumple 50, pero justo. El ajuste con e libre (40 caminantes, 12000 pasos) da 42 para b y 46 para c, por debajo de 50, así que sus resultados son orientativos.
8. **Línea base larga:** los sectores 3, 30 y 107 abarcan de 2018 a 2026, así que el periodo y T0 quedan muy correlacionados, sobre todo en c. Revisar trazas y autocorrelación, y si las cadenas divagan, acotar el prior del periodo con el valor publicado.
9. **Radio estelar:** se usa el valor publicado con su incertidumbre propagada.
10. **Masa de b:** sale 2.0σ por encima de la literatura (7.02 frente a 5.60 M⊕) porque el K de HARPS con LBL (≈ 4.8 m/s) es mayor que el publicado (3.80 m/s); el análisis de RV muestra un exceso de señal cerca de la rotación de la estrella (134 d).

### Referencias

- Cadieux et al. (2024), *New Mass and Radius Constraints on the LHS 1140 Planets: LHS 1140 b Is Either a Temperate Mini-Neptune or a Water World*, arXiv:2310.15490.
- Kipping, D. M. (2013), *Efficient, uninformative sampling of limb darkening coefficients for two-parameter laws*, MNRAS, 435, 2152.
- Mandel, K. & Agol, E. (2002), *Analytic Light Curves for Planetary Transit Searches*, ApJ, 580, L171. https://doi.org/10.1086/345520
- Parviainen, H. (2015), *PyTransit: fast and easy exoplanet transit modelling in Python*, MNRAS, 450, 3233. https://doi.org/10.1093/mnras/stv894
- Foreman-Mackey et al. (2013), *emcee: The MCMC Hammer*, PASP, 125, 306. https://doi.org/10.1086/670067
- Kopparapu, R. K. et al. (2013), *Habitable Zones around Main-sequence Stars: New Estimates*, ApJ, 765, 131.
- Lightkurve Collaboration (2018), *Lightkurve: Kepler and TESS time series analysis in Python*, Astrophysics Source Code Library, ascl:1812.013.
- Ricker, G. R. et al. (2015), *Transiting Exoplanet Survey Satellite (TESS)*, JATIS, 1, 014003.
- Jenkins, J. M. et al. (2016), *The TESS science processing operations center*, Proc. SPIE, 9913, 99133E.
- Astropy Collaboration (2022), *The Astropy Project: Sustaining and Growing a Community-oriented Open-source Project and the Latest Major Release (v5.0) of the Core Package*, ApJ, 935, 167.
- Datos: curvas de luz de TESS del archivo MAST (https://archive.stsci.edu).

**Fin del notebook.**